
# Smoker Status Prediction Using Bio-Signals — Logistic Regression

**Internal Practical Examination**

This notebook implements the required Logistic Regression workflow for predicting **smoking status** from biosignal measurements.

### Requirements covered
- Target: `smoking` (`0 = Non-Smoker`, `1 = Smoker`)
- Required final model: **Logistic Regression**
- Data preprocessing and leakage-safe validation
- Feature selection
- At least three meaningful derived/cross features
- Scaling experiment: no scaling, StandardScaler, MinMaxScaler
- Confusion matrix, Accuracy, Precision, Recall, F1-score and ROC-AUC
- Feature-crossing ablation
- Multicollinearity investigation
- Threshold optimization
- Logistic Regression coefficients and odds ratios
- Final comparison table and mandatory result analysis



## 1. Imports and reproducibility

The train/validation split is performed before fitting data-dependent preprocessing such as imputation, scaling and feature selection.


In [ ]:

import os
import glob
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, roc_curve
)

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
TEST_SIZE = 0.20
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")



## 2. Load the Kaggle dataset

The supplied Kaggle task URL requires authentication, so the notebook does not hard-code a private download URL. It automatically searches `/kaggle/input`, the current directory, and `/mnt/data` for a CSV containing the required `smoking` target.

If needed, set `DATA_PATH` manually.


In [ ]:

DATA_PATH = None
# Example:
# DATA_PATH = "/kaggle/input/<dataset-folder>/train.csv"

def find_smoking_csv():
    roots = ["/kaggle/input", ".", "/mnt/data"]
    candidates = []
    for root in roots:
        if os.path.exists(root):
            candidates.extend(glob.glob(os.path.join(root, "**", "*.csv"), recursive=True))

    candidates = list(dict.fromkeys(candidates))
    candidates = sorted(
        candidates,
        key=lambda p: (0 if "train" in os.path.basename(p).lower() else 1, len(p))
    )

    valid = []
    for path in candidates:
        try:
            sample = pd.read_csv(path, nrows=5)
            cols = {str(c).strip() for c in sample.columns}
            if "smoking" in cols:
                valid.append(path)
        except Exception:
            pass
    return valid

if DATA_PATH is None:
    matches = find_smoking_csv()
    if not matches:
        raise FileNotFoundError(
            "No CSV containing the 'smoking' target was found. "
            "Upload the Kaggle dataset or set DATA_PATH manually."
        )
    DATA_PATH = matches[0]

df = pd.read_csv(DATA_PATH)
df.columns = [str(c).strip() for c in df.columns]

print("Dataset:", DATA_PATH)
print("Shape:", df.shape)
display(df.head())



## 3. Data preprocessing and inspection

The supplied metadata identifies `id` as a unique record/person identifier and `smoking` as the target. The `id` column is excluded because it is not a meaningful biomedical predictor.

Potential outliers are reported with an IQR rule. They are **not automatically deleted**, because an IQR outlier is not necessarily an invalid biomedical measurement.


In [ ]:

TARGET = "smoking"
ID_COL = "id"

if TARGET not in df.columns:
    raise ValueError(f"Required target '{TARGET}' is missing.")

print("Data types:")
display(df.dtypes.to_frame("dtype"))

print("\nDuplicate rows:", df.duplicated().sum())

print("\nTarget distribution:")
target_counts = df[TARGET].value_counts(dropna=False).sort_index()
display(target_counts.to_frame("count").assign(proportion=target_counts / len(df)))

if ID_COL in df.columns:
    print("\nUnique IDs:", df[ID_COL].nunique(), "of", len(df))
    print("Decision: drop 'id' because it is a unique identifier, not a biomedical signal.")

print("\nMissing values:")
display(df.isna().sum().sort_values(ascending=False).head(20).to_frame("missing"))

print("\nDescriptive statistics:")
display(df.describe(include="all").T)


In [ ]:

numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
numeric_predictors = [c for c in numeric_cols if c not in [TARGET, ID_COL]]

outlier_rows = []
for col in numeric_predictors:
    s = df[col].dropna()
    if len(s) == 0:
        continue
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    if iqr == 0:
        n_out = 0
    else:
        low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        n_out = ((s < low) | (s > high)).sum()
    outlier_rows.append([col, q1, q3, iqr, n_out, n_out / len(s)])

outlier_report = pd.DataFrame(
    outlier_rows,
    columns=["feature", "Q1", "Q3", "IQR", "IQR_outliers", "outlier_rate"]
).sort_values("outlier_rate", ascending=False)

display(outlier_report)

print(
    "Outlier decision: investigate flagged observations; do not remove them solely "
    "because they fall outside the IQR range."
)



## 4. Leakage-safe train/validation split

The split happens before fitting the imputer, scaler and feature selector. Stratification preserves the smoker/non-smoker ratio.


In [ ]:

X = df.drop(columns=[TARGET], errors="ignore").copy()
y = df[TARGET].astype(int).copy()

X = X.drop(columns=[ID_COL], errors="ignore")
X = X.select_dtypes(include=np.number).copy()

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Training shape:", X_train.shape)
print("Validation shape:", X_val.shape)

print("\nTraining class proportions:")
display(y_train.value_counts(normalize=True).sort_index().to_frame("proportion"))

print("\nValidation class proportions:")
display(y_val.value_counts(normalize=True).sort_index().to_frame("proportion"))



## 5. Evaluation helpers


In [ ]:

def evaluate_model(model, Xtr, Xva, ytr, yva, threshold=0.50, name="Model"):
    model.fit(Xtr, ytr)
    prob = model.predict_proba(Xva)[:, 1]
    pred = (prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(yva, pred).ravel()

    return {
        "Experiment": name,
        "Threshold": threshold,
        "Accuracy": accuracy_score(yva, pred),
        "Precision": precision_score(yva, pred, zero_division=0),
        "Recall": recall_score(yva, pred, zero_division=0),
        "F1": f1_score(yva, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(yva, prob),
        "TN": tn, "FP": fp, "FN": fn, "TP": tp,
        "model": model, "prob": prob, "pred": pred
    }

def show_metrics(result):
    cols = ["Experiment", "Threshold", "Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]
    display(pd.DataFrame([{c: result[c] for c in cols}]).round(4))

def show_confusion(result, title):
    cm = np.array([[result["TN"], result["FP"]], [result["FN"], result["TP"]]])
    ConfusionMatrixDisplay(cm, display_labels=["Non-Smoker", "Smoker"]).plot()
    plt.title(title)
    plt.show()



## 6. Baseline Logistic Regression

Baseline uses the original numerical predictors, excluding `id`, with median imputation, StandardScaler and Logistic Regression.


In [ ]:

baseline_features = X_train.columns.tolist()

baseline_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(max_iter=3000, solver="lbfgs", random_state=RANDOM_STATE))
])

baseline_result = evaluate_model(
    baseline_model, X_train, X_val, y_train, y_val,
    threshold=0.50, name="Baseline"
)

show_metrics(baseline_result)
show_confusion(baseline_result, "Baseline Confusion Matrix")



## 7. Feature selection

Mutual information is used as the supervised feature-selection technique. It is fitted only on the training partition.

The selected feature set is compared with the original predictors.


In [ ]:

n_features = X_train.shape[1]
k_selected = min(n_features, max(5, int(np.ceil(0.60 * n_features))))

selector = SelectKBest(score_func=mutual_info_classif, k=k_selected)
X_train_sel = selector.fit_transform(X_train, y_train)
X_val_sel = selector.transform(X_val)

selected_features = X_train.columns[selector.get_support()].tolist()

selection_scores = pd.DataFrame({
    "feature": X_train.columns,
    "mutual_information": selector.scores_
}).sort_values("mutual_information", ascending=False)

print("Original feature count:", n_features)
print("Selected feature count:", len(selected_features))
print("\nSelected features:")
print(selected_features)
display(selection_scores)


In [ ]:

selected_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(max_iter=3000, solver="lbfgs", random_state=RANDOM_STATE))
])

selected_model.fit(X_train_sel, y_train)
selected_prob = selected_model.predict_proba(X_val_sel)[:, 1]
selected_pred = (selected_prob >= 0.50).astype(int)
cm = confusion_matrix(y_val, selected_pred).ravel()

selected_result = {
    "Experiment": "Feature Selected",
    "Threshold": 0.50,
    "Accuracy": accuracy_score(y_val, selected_pred),
    "Precision": precision_score(y_val, selected_pred, zero_division=0),
    "Recall": recall_score(y_val, selected_pred, zero_division=0),
    "F1": f1_score(y_val, selected_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_val, selected_prob),
    "TN": cm[0], "FP": cm[1], "FN": cm[2], "TP": cm[3],
    "model": selected_model, "prob": selected_prob, "pred": selected_pred
}

show_metrics(selected_result)



## 8. Meaningful feature crossing / engineering

The following derived features are created when their source columns are available:

1. **BMI** = weight / height² — relates body mass to height.
2. **Pulse pressure** = systolic − diastolic (`relaxation`) — summarizes blood-pressure difference.
3. **LDL/HDL ratio** — captures the balance between LDL and HDL.
4. **AST/ALT ratio** — summarizes the relative pattern of two liver enzymes.

These are meaningful biosignal relationships rather than arbitrary mathematical combinations.


In [ ]:

def add_cross_features(data):
    out = data.copy()
    created = []

    if {"height(cm)", "weight(kg)"}.issubset(out.columns):
        out["BMI"] = out["weight(kg)"] / (out["height(cm)"] / 100.0) ** 2
        created.append("BMI")

    if {"systolic", "relaxation"}.issubset(out.columns):
        out["pulse_pressure"] = out["systolic"] - out["relaxation"]
        created.append("pulse_pressure")

    if {"LDL", "HDL"}.issubset(out.columns):
        out["LDL_HDL_ratio"] = out["LDL"] / out["HDL"].replace(0, np.nan)
        created.append("LDL_HDL_ratio")

    if {"AST", "ALT"}.issubset(out.columns):
        out["AST_ALT_ratio"] = out["AST"] / out["ALT"].replace(0, np.nan)
        created.append("AST_ALT_ratio")

    out = out.replace([np.inf, -np.inf], np.nan)
    return out, created

X_cross, created_features = add_cross_features(X)

print("Created features:", created_features)
if created_features:
    display(X_cross[created_features].describe().T)


In [ ]:

Xc_train, Xc_val, yc_train, yc_val = train_test_split(
    X_cross, y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE
)

cross_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(max_iter=3000, solver="lbfgs", random_state=RANDOM_STATE))
])

cross_result = evaluate_model(
    cross_model, Xc_train, Xc_val, yc_train, yc_val,
    threshold=0.50, name="Cross Features"
)

show_metrics(cross_result)



### Feature-crossing ablation

Model A excludes crossed features. Model B includes them. ROC-AUC and F1 are compared on the same validation strategy.


In [ ]:

ablation = pd.DataFrame([
    {
        "Model": "A: Without Cross Features",
        "ROC-AUC": baseline_result["ROC-AUC"],
        "F1": baseline_result["F1"]
    },
    {
        "Model": "B: With Cross Features",
        "ROC-AUC": cross_result["ROC-AUC"],
        "F1": cross_result["F1"]
    }
])

display(ablation.round(4))
print("ROC-AUC change:", round(cross_result["ROC-AUC"] - baseline_result["ROC-AUC"], 4))
print("F1 change:", round(cross_result["F1"] - baseline_result["F1"], 4))



## 9. Scaling experiment

The same Logistic Regression model is evaluated with:
- No scaling
- StandardScaler
- MinMaxScaler

All preprocessing remains inside the pipeline.


In [ ]:

scalers = {
    "No Scaling": "passthrough",
    "StandardScaler": StandardScaler(),
    "MinMaxScaler": MinMaxScaler()
}

scaling_rows = []
scaling_models = {}

for name, scaler in scalers.items():
    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", scaler),
        ("logreg", LogisticRegression(max_iter=3000, solver="lbfgs", random_state=RANDOM_STATE))
    ])
    res = evaluate_model(pipe, X_train, X_val, y_train, y_val, 0.50, name)
    scaling_models[name] = res
    scaling_rows.append({
        "Scaling": name,
        "Accuracy": res["Accuracy"],
        "Precision": res["Precision"],
        "Recall": res["Recall"],
        "F1": res["F1"],
        "ROC-AUC": res["ROC-AUC"]
    })

scaling_results = pd.DataFrame(scaling_rows)
display(scaling_results.round(4))

best_scaling = scaling_results.sort_values(
    ["ROC-AUC", "F1"], ascending=False
).iloc[0]["Scaling"]
print("Best scaling option by ROC-AUC, F1 as tie-breaker:", best_scaling)



## 10. Multicollinearity investigation

Strongly correlated predictors are identified. Multicollinearity can make Logistic Regression coefficients unstable and harder to interpret even when prediction performance remains acceptable.


In [ ]:

corr = X_cross.corr(numeric_only=True)

upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
strong_pairs = (
    upper.stack()
    .reset_index()
    .rename(columns={"level_0": "Feature_1", "level_1": "Feature_2", 0: "Correlation"})
)
strong_pairs["Abs_Correlation"] = strong_pairs["Correlation"].abs()
strong_pairs = strong_pairs[strong_pairs["Abs_Correlation"] >= 0.80]     .sort_values("Abs_Correlation", ascending=False)

display(strong_pairs.head(25))

plt.figure(figsize=(12, 9))
plt.imshow(corr, aspect="auto")
plt.colorbar(label="Correlation")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.index)), corr.index)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()


In [ ]:

def choose_reduced_features(data, threshold=0.90):
    c = data.corr(numeric_only=True).abs()
    upper = c.where(np.triu(np.ones(c.shape), k=1).astype(bool))
    to_drop = [col for col in upper.columns if any(upper[col] > threshold)]
    return sorted(set(to_drop))

multicollinear_drop = choose_reduced_features(X_cross, threshold=0.90)
final_features = [c for c in X_cross.columns if c not in multicollinear_drop]

print("Features considered redundant at |r| > 0.90:")
print(multicollinear_drop)
print("\nFinal candidate feature count:", len(final_features))



## 11. Final Logistic Regression model

The final model uses meaningful engineered features and an evidence-based multicollinearity reduction. StandardScaler is used because it is a required scaling option and generally gives stable optimization for Logistic Regression.


In [ ]:

X_final = X_cross[final_features].copy()

Xf_train, Xf_val, yf_train, yf_val = train_test_split(
    X_final, y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE
)

final_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(max_iter=3000, solver="lbfgs", random_state=RANDOM_STATE))
])

final_result = evaluate_model(
    final_model, Xf_train, Xf_val, yf_train, yf_val,
    threshold=0.50, name="Final Model"
)

show_metrics(final_result)
show_confusion(final_result, "Final Model Confusion Matrix")



## 12. Threshold optimization

The default probability threshold of 0.50 is not automatically accepted. Several thresholds are evaluated.

The final threshold is selected using validation F1-score, with recall as a tie-breaker. This explicitly demonstrates the Precision/Recall trade-off.


In [ ]:

thresholds = np.arange(0.20, 0.81, 0.05)
threshold_rows = []

for t in thresholds:
    pred_t = (final_result["prob"] >= t).astype(int)
    threshold_rows.append({
        "Threshold": round(float(t), 2),
        "Precision": precision_score(yf_val, pred_t, zero_division=0),
        "Recall": recall_score(yf_val, pred_t, zero_division=0),
        "F1": f1_score(yf_val, pred_t, zero_division=0),
        "Accuracy": accuracy_score(yf_val, pred_t)
    })

threshold_table = pd.DataFrame(threshold_rows)
display(threshold_table.round(4))

best_threshold = float(
    threshold_table.sort_values(["F1", "Recall"], ascending=False).iloc[0]["Threshold"]
)

plt.figure(figsize=(9, 5))
plt.plot(threshold_table["Threshold"], threshold_table["Precision"], marker="o", label="Precision")
plt.plot(threshold_table["Threshold"], threshold_table["Recall"], marker="o", label="Recall")
plt.plot(threshold_table["Threshold"], threshold_table["F1"], marker="o", label="F1")
plt.xlabel("Probability Threshold")
plt.ylabel("Score")
plt.title("Threshold Optimization")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("Selected threshold:", best_threshold)

final_optimized_pred = (final_result["prob"] >= best_threshold).astype(int)
optimized_metrics = {
    "Threshold": best_threshold,
    "Accuracy": accuracy_score(yf_val, final_optimized_pred),
    "Precision": precision_score(yf_val, final_optimized_pred, zero_division=0),
    "Recall": recall_score(yf_val, final_optimized_pred, zero_division=0),
    "F1": f1_score(yf_val, final_optimized_pred, zero_division=0),
    "ROC-AUC": final_result["ROC-AUC"]
}
display(pd.DataFrame([optimized_metrics]).round(4))



## 13. ROC curve


In [ ]:

fpr, tpr, _ = roc_curve(yf_val, final_result["prob"])

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, label=f"Logistic Regression (AUC = {final_result['ROC-AUC']:.4f})")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — Final Logistic Regression")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()



## 14. Logistic Regression coefficients and odds ratios

For each standardized predictor:

**ORᵢ = exp(βᵢ)**

- OR > 1: higher predictor values are associated with higher modeled odds of smoker status.
- OR < 1: higher predictor values are associated with lower modeled odds.
- These are predictive associations, **not causal effects**.


In [ ]:

logreg = final_model.named_steps["logreg"]

coef_table = pd.DataFrame({
    "Feature": final_features,
    "Coefficient": logreg.coef_[0]
})
coef_table["Odds_Ratio"] = np.exp(coef_table["Coefficient"])
coef_table["Abs_Coefficient"] = coef_table["Coefficient"].abs()
coef_table = coef_table.sort_values("Abs_Coefficient", ascending=False)

print("Five most influential features:")
display(coef_table.head(5).round(4))

print("\nFull coefficient / odds-ratio table:")
display(coef_table.round(4))



## 15. Coefficient stability check

A second regularization strength is used to see whether important coefficients change substantially. Large changes can be a warning sign of instability or redundancy.


In [ ]:

def fit_coefficients(C_value):
    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("logreg", LogisticRegression(
            max_iter=3000, solver="lbfgs", C=C_value, random_state=RANDOM_STATE
        ))
    ])
    pipe.fit(Xf_train, yf_train)
    return pd.Series(pipe.named_steps["logreg"].coef_[0], index=final_features)

coef_c1 = fit_coefficients(1.0)
coef_c01 = fit_coefficients(0.1)

stability = pd.DataFrame({
    "C=1.0": coef_c1,
    "C=0.1": coef_c01
})
stability["absolute_change"] = (stability["C=1.0"] - stability["C=0.1"]).abs()

display(stability.sort_values("absolute_change", ascending=False).head(15).round(4))



## 16. Mandatory final result comparison

This table follows the requested structure: Baseline, Feature Selected, Cross Features and Final Model.


In [ ]:

comparison = pd.DataFrame([
    {
        "Experiment": "Baseline",
        "Features": len(baseline_features),
        "Scaling": "StandardScaler",
        "Threshold": 0.50,
        "Accuracy": baseline_result["Accuracy"],
        "Precision": baseline_result["Precision"],
        "Recall": baseline_result["Recall"],
        "F1": baseline_result["F1"],
        "ROC-AUC": baseline_result["ROC-AUC"]
    },
    {
        "Experiment": "Feature Selected",
        "Features": len(selected_features),
        "Scaling": "StandardScaler",
        "Threshold": 0.50,
        "Accuracy": selected_result["Accuracy"],
        "Precision": selected_result["Precision"],
        "Recall": selected_result["Recall"],
        "F1": selected_result["F1"],
        "ROC-AUC": selected_result["ROC-AUC"]
    },
    {
        "Experiment": "Cross Features",
        "Features": len(Xc_train.columns),
        "Scaling": "StandardScaler",
        "Threshold": 0.50,
        "Accuracy": cross_result["Accuracy"],
        "Precision": cross_result["Precision"],
        "Recall": cross_result["Recall"],
        "F1": cross_result["F1"],
        "ROC-AUC": cross_result["ROC-AUC"]
    },
    {
        "Experiment": "Final Model",
        "Features": len(final_features),
        "Scaling": "StandardScaler",
        "Threshold": best_threshold,
        "Accuracy": optimized_metrics["Accuracy"],
        "Precision": optimized_metrics["Precision"],
        "Recall": optimized_metrics["Recall"],
        "F1": optimized_metrics["F1"],
        "ROC-AUC": optimized_metrics["ROC-AUC"]
    }
])

display(comparison.round(4))



## 17. Mandatory result analysis

The next cell uses the measured validation results to identify which feature-engineering decision produced the largest observed change.


In [ ]:

changes = pd.DataFrame({
    "Decision": ["Feature Selection", "Feature Crossing"],
    "ROC-AUC Change": [
        selected_result["ROC-AUC"] - baseline_result["ROC-AUC"],
        cross_result["ROC-AUC"] - baseline_result["ROC-AUC"]
    ],
    "F1 Change": [
        selected_result["F1"] - baseline_result["F1"],
        cross_result["F1"] - baseline_result["F1"]
    ]
})

changes["Combined_Absolute_Change"] = (
    changes["ROC-AUC Change"].abs() + changes["F1 Change"].abs()
)

display(changes.sort_values("Combined_Absolute_Change", ascending=False).round(4))

best_decision = changes.sort_values(
    "Combined_Absolute_Change", ascending=False
).iloc[0]["Decision"]

print("1. Largest measured feature-engineering/preprocessing change:", best_decision)
print(
    "   Support this conclusion using the ROC-AUC and F1 changes displayed above."
)

top5 = coef_table.head(5)
print("\n2. Five most influential final features:")
display(top5[["Feature", "Coefficient", "Odds_Ratio"]].round(4))

print(
    "   OR > 1 means higher modeled odds of smoker status per standardized unit; "
    "OR < 1 means lower modeled odds, holding other predictors constant."
)

print("\n3. Main limitation:")
print(
    "   Logistic Regression models a linear relationship in log-odds and may not "
    "capture complex nonlinear relationships. The validation estimate also depends "
    "on the chosen train/validation split. Improvement could include repeated "
    "stratified cross-validation, better-calibrated threshold selection, and "
    "carefully justified nonlinear/interaction features while retaining Logistic Regression."
)



## 18. False positives and false negatives

For smoker prediction:

- **False Positive (FP):** a non-smoker is predicted as a smoker, which can cause unnecessary follow-up or incorrect classification.
- **False Negative (FN):** a smoker is predicted as a non-smoker, which can be more concerning in a screening context because an actual smoker may be missed.

The threshold therefore changes the balance between these two error types.


In [ ]:

tn, fp, fn, tp = confusion_matrix(yf_val, final_optimized_pred).ravel()

print("Final optimized-threshold confusion matrix:")
print(np.array([[tn, fp], [fn, tp]]))
print(f"True Negatives : {tn}")
print(f"False Positives: {fp}")
print(f"False Negatives: {fn}")
print(f"True Positives : {tp}")

if optimized_metrics["Recall"] > final_result["Recall"]:
    print("\nThe optimized threshold increased recall compared with threshold 0.50.")
elif optimized_metrics["Recall"] < final_result["Recall"]:
    print("\nThe optimized threshold decreased recall compared with threshold 0.50.")
else:
    print("\nRecall is unchanged compared with threshold 0.50.")



## 19. Submission checklist

Before submission, run all cells with the actual Kaggle dataset and verify that the final notebook contains:

- Data types, duplicates, target distribution and `id` decision
- Outlier investigation and justified treatment decision
- Feature-selection method and comparison
- At least three meaningful derived/cross features
- Leakage-safe scaling and Logistic Regression
- Confusion matrix, Accuracy, Precision, Recall, F1 and ROC-AUC
- Feature-crossing ablation
- Scaling experiment
- Multicollinearity investigation
- Threshold optimization
- Coefficients and odds ratios, including five influential features
- Final comparison table
- Explicit answers about the biggest improvement, influential biosignals and limitation
- Association vs causation distinction
